# Labor 3 Vertiefung: Huffman und Quellen mit Gedächtnis - Musterlösung

Dieses Notebook ist die Musterlösung zur Vertiefung für schnelle Gruppen.

Es baut einen einfachen Huffman-Encoder auf Byteebene und berechnet danach bedingte Entropien für dieselbe Textdatei `data/big.txt`.

Wichtig: Der Huffman-Vergleich berücksichtigt keine Speicherung der Code-Tabelle und kein Containerformat. Er zeigt die theoretische Wirkung variabler Codewortlängen, ist aber kein vollständiges Dateiformat.

## 1. Setup und Textdaten

Das Notebook nutzt dieselbe Datei wie das Pflichtnotebook: `./data/big.txt`.

Gearbeitet wird auf Byteebene. Ein Symbol ist also ein Bytewert zwischen `0` und `255`.


In [1]:
import gzip
import heapq
import itertools
import math
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

with open("./data/big.txt", "r", encoding="utf-8") as f:
    text = f.read()

text_data = text.encode("utf-8")
print(f"Zeichen im Text: {len(text):,}")
print(f"Bytes im Text:   {len(text_data):,}")
print(f"Verschiedene Bytewerte: {len(set(text_data))}")


Zeichen im Text: 6,488,666
Bytes im Text:   6,488,666
Verschiedene Bytewerte: 93


## 2. Hilfsfunktionen

Diese Funktionen werden später für Entropie und Vergleiche benötigt.

TODO: Ergänzen Sie die Entropieformel auf Basis von Häufigkeiten.


In [2]:
def entropie_aus_counts(counts):
    total = sum(counts.values())
    H = 0.0
    for count in counts.values():
        if count > 0:
            p = count / total
            H += -p * math.log2(p)
    return H

def gzip_size(data: bytes):
    return len(gzip.compress(data))

byte_counts = Counter(text_data)
H0 = entropie_aus_counts(byte_counts)
print(f"Entropie ohne Gedächtnis: {H0:.3f} bit/Byte")
print(f"Unkomprimiert: {8.0:.3f} bit/Byte")

Entropie ohne Gedächtnis: 4.511 bit/Byte
Unkomprimiert: 8.000 bit/Byte


# Teil A: Huffman-Encoder selbst bauen

Ein Huffman-Code entsteht durch wiederholtes Zusammenfassen der zwei seltensten Knoten.

In diesem Notebook werden Bäume sehr einfach dargestellt:

- Blatt: `("leaf", symbol)`
- innerer Knoten: `("node", left, right)`

TODO: Ergänzen Sie den Algorithmus so, dass aus Häufigkeiten ein Huffman-Baum entsteht.


In [3]:
def leaf(symbol):
    return ("leaf", symbol)

def node(left, right):
    return ("node", left, right)

def is_leaf(tree):
    return tree[0] == "leaf"

def build_huffman_tree(counts):
    tie = itertools.count()
    heap = []

    for symbol, weight in counts.items():
        heapq.heappush(heap, (weight, next(tie), leaf(symbol)))

    if len(heap) == 1:
        return heap[0][2]

    while len(heap) > 1:
        weight_left, _, left = heapq.heappop(heap)
        weight_right, _, right = heapq.heappop(heap)

        parent = node(left, right)
        parent_weight = weight_left + weight_right
        heapq.heappush(heap, (parent_weight, next(tie), parent))

    return heap[0][2]

huffman_tree = build_huffman_tree(byte_counts)
huffman_tree[0]

'node'

## 3. Codewörter aus dem Baum lesen

Der Weg nach links erhält hier eine `0`, der Weg nach rechts eine `1`.

TODO: Ergänzen Sie die rekursive Funktion.


In [4]:
def build_code_table(tree, prefix="", codes=None):
    if codes is None:
        codes = {}

    if is_leaf(tree):
        symbol = tree[1]
        codes[symbol] = prefix or "0"
        return codes

    _, left, right = tree
    build_code_table(left, prefix + "0", codes)
    build_code_table(right, prefix + "1", codes)
    return codes

codes = build_code_table(huffman_tree)
print(f"Codewörter: {len(codes)}")
print("Kürzeste Länge:", min(len(c) for c in codes.values()))
print("Längste Länge:", max(len(c) for c in codes.values()))

Codewörter: 93
Kürzeste Länge: 3
Längste Länge: 21


In [5]:
def byte_label(symbol):
    if 32 <= symbol <= 126:
        return repr(chr(symbol))
    return f"0x{symbol:02x}"

haeufigste = byte_counts.most_common(12)
for symbol, count in haeufigste:
    print(f"{byte_label(symbol):>6}  count={count:>8}  code={codes[symbol]}")


   ' '  count= 1036511  code=110
   'e'  count=  628234  code=000
   't'  count=  444459  code=1011
   'a'  count=  395872  code=1001
   'o'  count=  382683  code=0111
   'n'  count=  362397  code=0110
   'i'  count=  348464  code=0100
   's'  count=  326238  code=0011
   'r'  count=  303977  code=0010
   'h'  count=  287323  code=11111
   'd'  count=  211677  code=10100
   'l'  count=  195904  code=10001


## 4. Präfixfreiheit prüfen

Ein Huffman-Code muss präfixfrei sein.

TODO: Ergänzen Sie die Prüfung. Kein Codewort darf Anfang eines anderen Codeworts sein.


In [6]:
def ist_praefixfrei(code_table):
    codewords = list(code_table.values())
    for a in codewords:
        for b in codewords:
            if a != b and b.startswith(a):
                return False
    return True

print("präfixfrei:", ist_praefixfrei(codes))

präfixfrei: True


## 5. Encoding und Decoding

Für den Test wird nur ein Ausschnitt des Textes wirklich als Bitstring codiert. Für den gesamten Text wird später die Länge über Häufigkeiten berechnet.

TODO: Ergänzen Sie Encoder und Decoder.


In [7]:
def huffman_encode(data: bytes, code_table):
    return "".join(code_table[symbol] for symbol in data)

def huffman_decode(bitstring, tree):
    output = []
    current = tree
    for bit in bitstring:
        current = current[1] if bit == "0" else current[2]
        if is_leaf(current):
            output.append(current[1])
            current = tree
    return bytes(output)

sample = text_data[:20_000]
encoded_sample = huffman_encode(sample, codes)
decoded_sample = huffman_decode(encoded_sample, huffman_tree)

print("Bits im Sample:", len(encoded_sample))
print("Decoding korrekt:", decoded_sample == sample)

Bits im Sample: 93101
Decoding korrekt: True


## 6. Mittlere Codewortlänge und Größenvergleich

Die mittlere Codewortlänge ergibt sich aus den Symbolwahrscheinlichkeiten und den Codewortlängen.

TODO: Ergänzen Sie $L=\sum_i p_i l_i$.


In [8]:
def mittlere_codewortlaenge(counts, code_table):
    total = sum(counts.values())
    L = 0.0
    for symbol, count in counts.items():
        p = count / total
        L += p * len(code_table[symbol])
    return L

L = mittlere_codewortlaenge(byte_counts, codes)
huffman_bits_full = sum(count * len(codes[symbol]) for symbol, count in byte_counts.items())
huffman_bytes_without_table = math.ceil(huffman_bits_full / 8)

print(f"H0: {H0:.3f} bit/Byte")
print(f"L:  {L:.3f} bit/Byte")
print(f"Rohdaten:       {len(text_data):>10} Byte")
print(f"Huffman ohne Tabelle: {huffman_bytes_without_table:>10} Byte")
print(f"gzip:           {gzip_size(text_data):>10} Byte")
print(f"Abstand L-H0: {L-H0:.3f} bit/Byte")

H0: 4.511 bit/Byte
L:  4.541 bit/Byte
Rohdaten:          6488666 Byte
Huffman ohne Tabelle:    3683092 Byte


gzip:              2373530 Byte
Abstand L-H0: 0.030 bit/Byte


**Kurze Deutung zu Teil A:**

1. `L` ist nicht kleiner als `H0`, weil Entropie die untere Grenze der mittleren Codewortlänge für verlustfreie Codierung beschreibt. Der Huffman-Code liegt nah an dieser Grenze, kann sie aber wegen ganzzahliger Codewortlängen nicht beliebig exakt treffen.

2. Die Huffman-Größe ist keine vollständige Dateigröße, weil die Code-Tabelle, Blockgrenzen, Padding-Bits und ein Dateiformat nicht mitgezählt sind.

3. `gzip` kann besser sein, weil es nicht nur Einzelwahrscheinlichkeiten nutzt. Es erkennt Wiederholungen und längere Muster, bevor die verbleibenden Daten entropiecodiert werden.

# Teil B: Bedingte Entropie und Quelle mit Gedächtnis 2. Ordnung

Bisher wurde der Text wie eine gedächtnislose Quelle betrachtet: Jedes Byte wurde nur über seine Einzelhäufigkeit modelliert.

Jetzt wird gefragt:

- Wie unsicher ist das nächste Byte, wenn das vorherige Byte bekannt ist?
- Wie unsicher ist das nächste Byte, wenn die vorherigen zwei Bytes bekannt sind?

Für Ordnung 2 lautet die Größe:

$$
H(X_n \mid X_{n-2}, X_{n-1}).
$$


## 7. Folgezeichen pro Kontext zählen

Ein Kontext der Ordnung 2 besteht aus den zwei vorherigen Bytes.

Beispiel: Wenn im Text oft `th` vorkommt, ist das nächste Zeichen nicht gleichverteilt.

TODO: Ergänzen Sie die Zählung der Folgezeichen.


In [9]:
def follower_counts(sequence: bytes, order: int):
    followers = defaultdict(Counter)
    for i in range(order, len(sequence)):
        context = tuple(sequence[i-order:i])
        next_symbol = sequence[i]
        followers[context][next_symbol] += 1
    return followers

followers_order_1 = follower_counts(text_data, order=1)
followers_order_2 = follower_counts(text_data, order=2)

print("Kontexte Ordnung 1:", len(followers_order_1))
print("Kontexte Ordnung 2:", len(followers_order_2))

Kontexte Ordnung 1: 93
Kontexte Ordnung 2: 2870


## 8. Bedingte Entropie berechnen

Die bedingte Entropie ist ein gewichteter Mittelwert über alle Kontexte:

$$
H(X_n \mid Kontext)=\sum_c P(c)H(X_n\mid c).
$$

TODO: Ergänzen Sie die Gewichtung der Kontextentropien.


In [10]:
def conditional_entropy(sequence: bytes, order: int):
    followers = follower_counts(sequence, order)
    total_context_occurrences = sum(sum(counts.values()) for counts in followers.values())
    H_cond = 0.0

    for context, counts in followers.items():
        context_count = sum(counts.values())
        H_given_context = entropie_aus_counts(counts)
        weight = context_count / total_context_occurrences
        H_cond += weight * H_given_context

    return H_cond

H1 = conditional_entropy(text_data, order=1)
H2 = conditional_entropy(text_data, order=2)

print(f"H(X_n)                         = {H0:.3f} bit/Byte")
print(f"H(X_n | X_(n-1))               = {H1:.3f} bit/Byte")
print(f"H(X_n | X_(n-2), X_(n-1))      = {H2:.3f} bit/Byte")
print(f"Reduktion von H0 zu H2:          {H0-H2:.3f} bit/Byte")

H(X_n)                         = 4.511 bit/Byte
H(X_n | X_(n-1))               = 3.613 bit/Byte
H(X_n | X_(n-2), X_(n-1))      = 2.921 bit/Byte
Reduktion von H0 zu H2:          1.590 bit/Byte


## 9. Häufige Kontexte ansehen

Nicht alle Kontexte sind gleich informativ. Häufige Kontexte zeigen oft gut, warum Sprache Gedächtnis besitzt.


In [11]:
def context_label(context):
    return bytes(context).decode("utf-8", errors="replace")

def symbol_label(symbol):
    return bytes([symbol]).decode("utf-8", errors="replace")

haeufige_kontexte = sorted(
    followers_order_2.items(),
    key=lambda item: sum(item[1].values()),
    reverse=True
)[:10]

for context, counts in haeufige_kontexte:
    top_followers = counts.most_common(5)
    followers_text = ", ".join(f"{symbol_label(s)!r}:{c}" for s, c in top_followers)
    print(f"Kontext {context_label(context)!r:>8} -> {followers_text}")


Kontext     'e ' -> 't':17392, 'a':14825, 's':14533, 'o':13561, 'w':11295
Kontext     ' t' -> 'h':102816, 'o':30798, 'r':3955, 'e':3130, 'i':3116
Kontext     'he' -> ' ':84632, 'r':18056, '\n':8425, 'n':4920, 'y':3942
Kontext     'th' -> 'e':92805, 'a':13367, ' ':11695, 'i':8166, 'o':5006
Kontext     ' a' -> 'n':42096, ' ':17581, 's':9525, 't':7378, 'l':7231
Kontext     'd ' -> 't':20401, 'a':11201, 'h':7722, 'b':7445, 'i':6902
Kontext     's ' -> 'a':14056, 'o':11987, 't':10760, 'i':7004, 'w':6576
Kontext     'in' -> 'g':35008, ' ':22169, 't':6498, 'c':5198, 'e':4722
Kontext     'er' -> ' ':21283, 'e':14478, 's':8725, 'i':4922, 'a':4364
Kontext     't ' -> 't':13646, 'o':7997, 'a':7912, 'i':7043, 'w':6355


**Kurze Deutung zu Teil B:**

1. Zusätzlicher Kontext kann Unsicherheit senken, weil das nächste Symbol durch vorherige Symbole besser vorhersagbar wird. Deshalb sollte $H(X_n \mid X_{n-2},X_{n-1}) \le H(X_n \mid X_{n-1}) \le H(X_n)$ gelten.

2. Häufige Zwei-Byte-Kontexte zeigen typische Sprach- und Textmuster, zum Beispiel Leerzeichen plus Buchstabe oder häufige Buchstabenfolgen. Die Folgezeichen sind dann nicht mehr gleichverteilt.

3. Eine Quelle mit Gedächtnis erklärt natürliche Sprache besser, weil Buchstaben und Bytes nicht unabhängig auftreten. Aus dem Kontext entstehen Vorhersagbarkeit und damit Komprimierbarkeit.

## 10. Ergebnis für schnelle Gruppen

Die konkreten Werte hängen von `data/big.txt` ab. Für die aktuelle Datei ergibt sich ungefähr:

- Der selbst gebaute Huffman-Code erreicht `L = 4.541 bit/Byte`.
- Die Entropie ohne Gedächtnis beträgt `H0 = 4.511 bit/Byte`.
- Die bedingte Entropie erster Ordnung beträgt `H1 = 3.613 bit/Byte`.
- Die bedingte Entropie zweiter Ordnung beträgt `H2 = 2.921 bit/Byte`.
- Der Unterschied zwischen Huffman und Quelle mit Gedächtnis ist: Huffman auf Einzelbytes nutzt nur Einzelhäufigkeiten; die bedingte Entropie nutzt zusätzlich Kontext und zeigt deshalb weiteres Kompressionspotenzial.
- Eine anspruchsvolle Klausurfrage wäre: Warum kann ein Modell mit Gedächtnis eine niedrigere Entropie liefern als eine gedächtnislose Quelle, obwohl dieselbe Datei betrachtet wird?